In [0]:
dbutils.widgets.text(
    "config_path",
    ""
)

In [0]:
config_path = dbutils.widgets.get("config_path")


print(f"Reading config from: {config_path}")

config_df = (
    spark.read
         .option("header", True)
         .csv(config_path)
)

In [0]:
table_configs = [
    {
        # Core fields
        "data_source": row_dict["data_source"],
        "report_type": row_dict["validation"],              # CSV: validation → report_type
        "secret_scope": row_dict["scope_name"],             # CSV: scope_name → secret_scope

        # Source fields
        "source_catalog": row_dict["source_catalog"],
        "source_schema": row_dict["source_schema"],
        "source_table": row_dict["source_table"],

        # Target fields
        "target_catalog": row_dict["target_catalog"],
        "target_schema": row_dict["target_schema"],
        "target_table": row_dict["target_table_name"],      # CSV: target_table_name → target_table

        # Join columns
        "join_columns": row_dict["primary_key"].split(","), # CSV: primary_key → join_columns (split)

        # Optional filter fields
        "filters_column": row_dict.get("filters_column"),
        "filters_column_type": row_dict.get("filters_column_type", ""),  # NEW: Added for dynamic filtering
        "source_filters_condition": row_dict.get("source_filters_condition"),
        "target_filters_condition": row_dict.get("target_filters_condition"),
        
        # Optional transformation fields
        "transformation": row_dict.get("transformation"),
        "column_mapping": row_dict.get("column_mapping")
    }
    for row in config_df.collect()
    for row_dict in [row.asDict()]  # Convert Row to dict for safe .get() access
]

In [0]:
dbutils.jobs.taskValues.set(
    key="recon_config",
    value=table_configs
)

In [0]:
print(f" Loaded {len(table_configs)} table configuration(s)")
print("\nConfiguration summary:")
for config in table_configs:
    print(f"  - {config['source_table']} → {config['target_table']} ({config['report_type']})")